# F1 — Definición del proyecto
**MCDI500 · Programación para la Ciencia de Datos**

Integrantes:
- Matías Manríquez Ortiz
- Daniel Pérez Ramirez
- Abigail Robles Chávez
- Roberto Sánchez Saldivia

> **Alcance de este cuaderno:** define el problema, las preguntas, los objetivos y el alcance del proyecto, y verifica que el entorno reproducible esté funcionando antes de avanzar a la Fase 2.

## 1. Contexto y formulación del problema

**Dataset:** [2023 National Youth Risk Behavior Survey (YRBS)](https://www.cdc.gov/yrbs/data/index.html) — Centers for Disease Control and Prevention (CDC).

El proyecto originalmente usó el *Mental Health and Technology Usage Dataset* de Kaggle. Se reemplazó porque sus variables categóricas se repartían en proporciones perfectamente parejas (evidencia de datos simulados) y un análisis publicado sobre ese mismo conjunto mostró coeficientes cercanos a cero entre todas las variables de interés: la pregunta de investigación no tenía respuesta posible con esos datos. El detalle de esta decisión está registrado en `docs/bitacora_decisiones.md`.

El YRBS es una encuesta real del CDC aplicada a estudiantes de enseñanza media en Estados Unidos. La versión nacional 2023 usada aquí tiene 20.103 registros y 117 variables, con diseño muestral complejo (`weight`, `stratum`, `psu`).

## 2. Pregunta de investigación

**Pregunta principal:**
¿Qué relación existe entre los patrones de uso de tecnología y los indicadores de salud mental autopercibida y horas de sueño en los estudiantes incluidos en el dataset YRBS 2023, considerando variables sociodemográficas y de actividad física?

**Variables:**
- Desenlace: `salud_mental_cod` (Q84 — salud mental no buena en los últimos 30 días; ordinal, 5 niveles: Nunca–Siempre)
- Explicativas: `redes_sociales_cod` (Q80, 8 niveles), `sueno_cod` (Q85, 7 niveles), `actividad_fisica_cod` (Q76, 0-7 días)
- Covariables demográficas: `edad_cod` (Q1), `sexo_cod` (Q2)

Todas las variables y su codificación se documentan con su fuente oficial (codebook CDC, Apéndice C) en `src/procesamiento.py`.

## 3. Objetivo general

Analizar la relación entre los patrones de uso de tecnología (dispositivos electrónicos y redes sociales) y los indicadores de salud mental autopercibida y horas de sueño en los estudiantes incluidos en el dataset YRBS 2023, controlando por variables sociodemográficas y de actividad física, mediante un flujo de trabajo reproducible y documentado.

## 4. Objetivos específicos

1. Caracterizar la distribución sociodemográfica y los patrones iniciales de uso de tecnología, salud mental y sueño en la muestra, mediante análisis exploratorio descriptivo para establecer la línea de base del estudio. 
2. Diagnosticar y preprocesar el conjunto de datos YRBS 2023 mediante un pipeline modular en Python, identificando y tratando valores faltantes, atípicos e inconsistencias para garantizar la integridad y calidad analítica de la data.
3. Validar el dataset resultando mediante pruebas de consistencia técnica y verificación estructural con assert, asegurando la exportación de un conjunto limpio a data/processed/ para las fases posteriores.
4. Explorar y evaluar las asociaciones preliminares entre las variables tecnológicas, de estilo de vida y salud mental, preparando la estructura de datos para el posterior modelamiento estadístico y regresional en las Fases 3 y 4.


## 5. Alcance, restricciones y supuestos

**Alcance:** definición del problema, configuración del entorno y desarrollo del pipeline de datos (Fases 1-2). No incluye modelación predictiva ni pruebas de hipótesis formales.

**Supuestos:**
- Las respuestas por los estudiantes reflejan su percepción, no un diagnóstico clínico.
- Las 5 variables de análisis (Q1, Q2, Q76, Q80, Q84, Q85, raceeth) no tienen dependencia de otra pregunta en el codebook oficial: sus valores nulos se interpretan como no respuesta genuina, no como salto de pregunta.

**Restricción declarada:** este proyecto **no pondera** con `weight`/`stratum`/`psu` en las Fases 1-2. En consecuencia, todo resultado describe la muestra de 20.103 estudiantes encuestados en 2023, y no se generaliza a la población de estudiantes de EE. UU.

## 6. Configuración y verificación del entorno
Código base que evidencia que el entorno reproducible está funcionando.

In [1]:
import sys                       # intérprete en uso: es lo que se verifica más abajo
import json                      # exportación de metadatos legibles
import platform                  # sistema operativo, para dejarlo en la bitácora
import subprocess                # consultas a Git desde el cuaderno
import shutil                    # localizar ejecutables (git) sin suponer que existen
import importlib                 # importar el módulo que este cuaderno va a escribir
from pathlib import Path         # manejo de rutas independiente del sistema operativo
from datetime import date

DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. Intérprete que ejecuta ESTE cuaderno. Si la ruta no contiene .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde aquí,
    #    no desde donde está guardado el archivo .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se interrumpe el cuaderno: se informa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)

Intérprete       : D:\Estudio\Magíster en Ciencia de Datos e Inteligencia Artificial - UNAB\MCDI500 Programación para la Ciencia de Datos\proyecto-grupo8-mcdi500\.venv\Scripts\python.exe
Entorno virtual  : [OK] activo
Carpeta de trabajo: C:\Users\el_ma\Desktop\cuadernos\proyecto-grupo8-mcdi500\F1\notebooks
Sistema          : Windows 11
Python           : 3.13.15

Librerías declaradas
  [OK]    numpy        2.5.3
  [OK]    pandas       3.0.6
  [OK]    matplotlib   3.11.2
  [OK]    sklearn      1.9.1


## 7. Reconocimiento inicial del conjunto de datos
Solo carga y una mirada superficial — sin seleccionar columnas, limpiar ni transformar (eso ocurre en `F2/S1_F2_Preprocesamiento.ipynb`, usando las funciones de `src/procesamiento.py`).

In [2]:
import sys
from pathlib import Path

# Agrega src/ de la raiz del proyecto al path
sys.path.append(str(Path('..') / '..' / 'src'))
from procesamiento import cargar_datos

# Ruta relativa al notebook: F1/notebooks/ -> F1/data/raw/
RUTA = Path('..') / 'data' / 'raw' / 'XXH2023_YRBSS_data.csv'
df = cargar_datos(str(RUTA))

print('Filas, columnas:', df.shape)
df.head()

Filas, columnas: (20103, 117)


,site,raceeth,q6orig,q7orig,record,orig_rec,q1,q2,q3,q4,...,q102,q103,q104,q105,q106,q107,BMIPCT,weight,stratum,psu
0,XX,NaN,505,180,1,NaN,3.0,1.0,1.0,NaN,...,2.0,3.0,3.0,2.0,2.0,2.0,97.083855,0.8614,103,16294
1,XX,5.0,N N,233,2,NaN,4.0,2.0,1.0,2.0,...,2.0,2.0,5.0,2.0,1.0,2.0,NaN,0.8920,103,16294
2,XX,5.0,506,165,3,NaN,5.0,2.0,3.0,2.0,...,1.0,2.0,4.0,2.0,2.0,1.0,92.262516,0.5081,103,16294
3,XX,5.0,N N,105,4,NaN,6.0,1.0,2.0,2.0,...,2.0,4.0,5.0,2.0,1.0,1.0,NaN,1.1759,103,16294
4,XX,5.0,601,125,5,NaN,3.0,2.0,1.0,2.0,...,2.0,2.0,5.0,2.0,2.0,1.0,7.567053,0.8920,103,16294


## 8. Vinculación con el mapa conceptual (breve)

Los bloques 1 a 8 del mapa conceptual se materializan en el avance de las Fases 1 y 2: el bloque 1 (Problema y preguntas) en las secciones 1-5 de este notebook; el bloque 2 (Datos) en la carga anterior y en el diagnóstico completo de `S1_F2_Preprocesamiento.ipynb`; el bloque 3 (Entorno reproducible) en la verificación de la sección 6. La modelación y la comunicación visual de resultados, proyectadas en el mapa como trabajo de fases posteriores, se abordarán en las Fases 3 y 4.

---
**Antes de entregar:** `Kernel → Restart Kernel and Run All Cells` y verificar que corre sin errores de inicio a fin.